# RAG (Retrieval-Augmented Generation)

A lightweight Retrieval-Augmented Generation (RAG) pipeline built in Google Colab.

### What it does

* Upload PDF, DOCX, or TXT files
* Extract and chunk text
* Create embeddings with FAISS indexing
* Generate summaries
* Perform semantic search
* Create document-specific FAQs



## 1. Setup & Installation

Install the third-party libraries this notebook depends on:

- `transformers`, `sentence-transformers` — embedding & generation models
- `faiss-cpu` — vector similarity search
- `pypdf`, `python-docx`, `pymupdf` (`fitz`) — document parsing
- `nltk` — sentence tokenization
- `gradio` — (imported for potential UI use)

project uses `transformers==4.52.4` and `sentence-transformers==4.1.0` because these versions provide stable support for the summarization and text2text-generation pipeline tasks

In [ ]:
!pip install -q transformers==4.52.4 sentence-transformers==4.1.0 faiss-cpu pypdf python-docx nltk gradio pymupdf

In [ ]:
# import os
# os.kill(os.getpid(), 9)

## 2. Imports

Loads all required standard-library and third-party packages used across the notebook, including document processing, text preprocessing, embeddings, vector search, and LLM-based generation. It also downloads the NLTK tokenizer resources needed for sentence splitting.
.


In [ ]:
import os, io, math, gc, json, textwrap, re, random, collections

from dataclasses import dataclass
from typing import List, Dict, Tuple

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

from google.colab import files
from datetime import datetime

from pypdf import PdfReader
from docx import Document as DocxDocument


from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

import fitz  # PyMuPDF
import faiss
import numpy as np
import gradio as gr
import torch


## 3. Configuration

All tunable constants, model names, and prompt templates live here so they
are easy to find and change in one place, instead of being scattered across
the notebook.



In [ ]:
# ---- Device ----
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"

Added Generic prompt-leak pattern removal using regular expressions (re) to filter instruction templates, prompt, and generation metadata from model outputs.

In [ ]:
device_id = 0 if DEVICE == "cuda" else -1

# File upload constraints
max_file = 5
max_file_size = 15  # MB
file_extensions = {".pdf", ".docx", ".txt"}

# Chunking
DEFAULT_CHUNK_CHARS = 1400
DEFAULT_OVERLAP_SENTS = 1

# Embedding model
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_BATCH_SIZE = 64

# Summarization / Generation models
SUMM_MODEL = "facebook/bart-large-cnn"
GEN_MODEL = "google/flan-t5-base"

# Prompt templates
ANSWER_PROMPT_TMPL = """You are a helpful assistant. Answer the question strictly using the provided context.
If the answer is not in the context, say "I don't know from the provided documents."

Context:
{context}

Question: {question}

Answer:"""

FAQ_QS_PROMPT = """Write 12 distinct, specific FAQ questions that a reader of the following document would likely ask.
Questions must reference concrete people/organizations, locations, years/dates, metrics, methods, or outcomes from the text.
Avoid generic questions like "What is the main point?" or "Why is this important?".

Return ONLY the questions, one per line, no numbering or extra text.

Document:
\"\"\"{doc}\"\"\""""

SUMMARIZATION_PROMPT = """Summarize the document below into 7–10 concise bullet points.
Each bullet must be specific and factual, and should include named entities (people, orgs, places), dates/years,
numbers/tables findings if present, and concrete outcomes or claims. Avoid generic sentences.

Document:
{joined}

Return only bullet points, each starting with '-'."""


# Generic / low-value FAQ question patterns (used to filter out vague questions)
GENERIC_PATTERNS = [
    r"\bmain point\b",
    r"\bwhy .* important\b",
    r"\bintended audience\b",
    r"\bscope\b",
    r"\bconclusion\b",
    r"\boverview\b",
    r"\bsum(mary|marize)\b",
    r"\badditional detail\??\b",
    r"\bwhat is .*document\b",
    r"\bwhat does the document\b",
]
GENERIC_RE = re.compile("|".join(GENERIC_PATTERNS), re.I)

# Generic prompt leak patterns ( used to remove prompt from output)

PROMPT_LEAK_PATTERNS = [
    r"\breturn only\b",
    r"\bone per line\b",
    r"\bno numbering\b",
    r"\bsummarize the (document|text) below\b",
    r"\bwrite \d+ distinct\b",
    r"^document:\s*$",
    r"^context:\s*$",
    r"^question:\s*$",
    r"^answer:\s*$",
    r'^"""',
]
PROMPT_LEAK_RE = re.compile("|".join(PROMPT_LEAK_PATTERNS), re.I)



## 4. Data Loading

### 4.1 Upload documents

Upload up to `max_file` documents (PDF / DOCX / TXT, each under
`max_file_size` MB).

In [ ]:
print(f"Please upload max {max_file} files of max size {max_file_size} MB with these extensions {file_extensions}")

uploaded = files.upload()

def ext_of(name):
    return os.path.splitext(name)[1].lower()

assert len(uploaded) > 0, "No files uploaded."
assert len(uploaded) <= max_file, f"Please upload at most {max_file} files."

docs_raw = []

for fname, b in uploaded.items():
    assert ext_of(fname) in file_extensions, f"Unsupported file type for {fname}. Allowed: {file_extensions}"

    size_of = len(b) / (1024 * 1024)
    assert size_of <= max_file_size, f"{fname} is {size_of:.2f} MB, exceeds {max_file_size} MB."

    docs_raw.append((fname, b))

print("Uploaded files:", [d[0] for d in docs_raw])

### 4.2 File readers

Helper functions to extract raw text from each supported file type, and a
dispatcher (`load_text_by_ext`) that picks the right reader based on file
extension.

In [ ]:
def read_txt(bytes_blob: bytes) -> str:
    return io.BytesIO(bytes_blob).read().decode('utf-8', errors='ignore')

def read_pdf(bytes_blob: bytes) -> str:
    reader = PdfReader(io.BytesIO(bytes_blob))
    texts = []
    for page in reader.pages:
        try:
            texts.append(page.extract_text() or "")
        except:
            texts.append("")
    return "\n".join(texts)

def read_docx(bytes_blob: bytes) -> str:
    fh = io.BytesIO(bytes_blob)
    doc = DocxDocument(fh)
    return "\n".join([p.text for p in doc.paragraphs])

def load_text_by_ext(fname: str, blob: bytes) -> str:
    ext = os.path.splitext(fname)[1].lower()

    match ext:
        case ".txt":
            return read_txt(blob)

        case ".pdf":
            return read_pdf(blob)

        case ".docx":
            return read_docx(blob)

### 4.3 Extract & clean text

Load the text for every uploaded document and apply light whitespace
cleanup. `_clean_text` is a small string-normalization helper reused later
during title extraction.

In [ ]:
def _clean_text(s: str) -> str:
    s = re.sub(r'\s+', ' ', s).strip()
    s = re.sub(r'^\W+|\W+$', '', s).strip()
    return s

docs_text = []
for fname, blob in docs_raw:
    text = load_text_by_ext(fname, blob)
    # light cleanup
    text = re.sub(r'\s+\n', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip()
    docs_text.append({"name": fname, "text": text})

for d in docs_text:
    print(d["name"], "characters:", len(d["text"]))

## 5. Preprocessing — Title Extraction

Each document's display title is detected automatically from its layout
(largest/boldest text near the top of the first pages for PDFs, the
highest-priority heading style for DOCX, or the first non-empty line for
TXT). If no title can be detected, the filename is used as a fallback.

In [ ]:
def extract_title_from_pdf(pdf_bytes: bytes, consider_pages=3, top_band=0.35):
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    if doc.page_count == 0: return None

    spans, page_heights = [], {}
    for pno in range(min(consider_pages, doc.page_count)):
        pg = doc.load_page(pno)
        page_heights[pno] = pg.rect.height
        data = pg.get_text("dict")
        for b in data.get("blocks", []):
            for l in b.get("lines", []):
                for s in l.get("spans", []):
                    txt = _clean_text(s.get("text",""))
                    if not txt or len(txt) < 3: continue
                    size = float(s.get("size", 0))
                    font = (s.get("font") or "").lower()
                    is_bold = ("bold" in font) or ("black" in font)
                    y0 = s.get("bbox", [0,0,0,0])[1]
                    spans.append({"page": pno, "text": txt, "size": size, "bold": is_bold, "y0": y0})
                    # print(spans)
    if not spans: return None

    # keep top-of-page band only
    top_spans = [sp for sp in spans if page_heights.get(sp["page"],1) and (sp["y0"]/page_heights[sp["page"]]) <= top_band]
    spans = top_spans or spans

    # drop strings repeating on >=2 pages (likely header)
    per_text_pages = collections.defaultdict(set)
    for sp in spans: per_text_pages[sp["text"]].add(sp["page"])
    spans = [sp for sp in spans if not (len(per_text_pages[sp["text"]]) >= 2 and len(sp["text"]) <= 80)] or spans

    # group consecutive lines with same size+bold on same page to capture multi-line titles
    groups = []
    spans.sort(key=lambda x: (x["page"], -x["size"], x["y0"]))
    for sp in spans:
        placed = False
        for g in groups:
            if g["page"] == sp["page"] and abs(g["size"]-sp["size"]) < 0.5 and g["bold"] == sp["bold"]:
                if abs(sp["y0"] - g["lines"][-1][1]) < 22:  # vertically close
                    g["lines"].append((sp["text"], sp["y0"]))
                    placed = True
                    break
        if not placed:
            groups.append({"page": sp["page"], "size": sp["size"], "bold": sp["bold"], "lines": [(sp["text"], sp["y0"])]})

    groups.sort(key=lambda g: (g["size"], g["bold"], -len(g["lines"]), -1.0/min([y for _,y in g["lines"]] or [1])), reverse=True)
    for g in groups:
        lines_sorted = [t for t,_ in sorted(g["lines"], key=lambda x: x[1])]
        cand = _clean_text(" ".join(lines_sorted))
        if 5 <= len(cand) <= 200:
            return cand

    best = max(spans, key=lambda sp: (sp["size"], sp["bold"], -sp["y0"]))
    return best["text"] if best else None

In [ ]:
def extract_title_from_docx(docx_bytes: bytes):
    d = DocxDocument(io.BytesIO(docx_bytes))
    def paragraph_score(p):
        text = _clean_text(p.text)
        if not text or len(text) < 3: return None
        style_name = (p.style.name if p.style else "").lower()
        max_size_pt, any_bold = 0.0, False
        for run in p.runs:
            if run.font is not None:
                if run.font.size:
                    try: max_size_pt = max(max_size_pt, float(run.font.size.pt))
                    except: pass
                if run.font.bold: any_bold = True
        style_priority = 0
        if "title" in style_name: style_priority = 3
        elif "heading 1" in style_name or style_name == "heading1": style_priority = 2
        elif "heading" in style_name: style_priority = 1
        return {"text": text, "stylePriority": style_priority, "size": max_size_pt, "bold": any_bold}

    candidates = []
    for p in d.paragraphs:
        sc = paragraph_score(p)
        if sc: candidates.append(sc)
    if not candidates: return None
    candidates.sort(key=lambda c: (c["stylePriority"], c["size"], c["bold"], -len(c["text"])) , reverse=True)
    return candidates[0]["text"]


def extract_title_from_txt(txt: str):
    for line in txt.splitlines():
        if line.strip(): return _clean_text(line)
    return None

Apply the right title-extraction function to each document based on its
file extension, falling back to the filename (without extension) when no
title can be detected.

In [ ]:
name_to_title = {}
name_to_blob = {file_name: blob for file_name, blob in docs_raw}

for item in docs_text:
    file_name = item["name"]
    blob = name_to_blob[file_name]

    ext = os.path.splitext(file_name)[1].lower()
    title = None

    try:
        match ext:
            case ".pdf":
                title = extract_title_from_pdf(blob)

            case ".docx":
                title = extract_title_from_docx(blob)

            case ".txt":
                txt = read_txt(blob)
                title = extract_title_from_txt(txt)

    except Exception:
        title = None

    if not title:
        title = os.path.splitext(file_name)[0]

    name_to_title[file_name] = title

print("Detected titles:")

for doc in docs_text:
    print(f"{doc['name']} → {name_to_title[doc['name']]}")

## 6. Preprocessing — Chunking

Split each document's text into overlapping, sentence-aware chunks. Chunks
are built up sentence-by-sentence until they reach `target_chars`, with the
last `overlap_sent` sentences of a chunk carried over into the next one so
context isn't lost at chunk boundaries.

In [ ]:
def chunk_text(text: str, target_chars: int = 1200, overlap_sent: int = 2) -> List[str]:
    sents = sent_tokenize(text)
    assert overlap_sent >= 0, "overlap_sent must be >= 0"
    assert overlap_sent <= 5, (
        f"overlap_sent={overlap_sent} looks too large (expected a small sentence count, "
        f"not a character count) — check the caller."
    )
    chunks = []
    buf_sents = []
    buf_len = 0

    for s in sents:
        if not buf_sents:
            buf_sents.append(s)
            buf_len = len(s)

        elif buf_len + 1 + len(s) <= target_chars:
            buf_sents.append(s)
            buf_len += 1 + len(s)
        # overlap tail
        else:
            chunks.append(" ".join(buf_sents).strip())
            # overlap last N sentences
            overlap = buf_sents[-overlap_sent:] if overlap_sent > 0 else []

            buf_sents = overlap + [s]
            buf_len = sum(len(x) for x in buf_sents) + max(0, len(buf_sents) - 1)
    # filter tiny empties
    if buf_sents:
        chunks.append(" ".join(buf_sents).strip())

    return [c for c in chunks if len(c.strip()) > 0]


all_docs_chunks = []

for d in docs_text:
    ch = chunk_text(d["text"] , target_chars=1400 , overlap_sent=1 )

    all_docs_chunks.append({ "name": d["name"] , "chunks": ch })

    print(d["name"], "→", len(ch), "chunks")

## 7. Model Setup

### 7.1 Embeddings & FAISS index

Embed every chunk with the sentence-transformer model and build a FAISS
flat index for cosine-similarity search (vectors are L2-normalized, so an
inner-product index is equivalent to cosine similarity).

In [ ]:
embedder = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)

# Flatten chunks + keep metadata
corpus_texts, corpus_meta = [], []  # meta: (doc_id, doc_name, chunk_id)
for doc_id, d in enumerate(all_docs_chunks):
    for chunk_id, c in enumerate(d["chunks"]):
        corpus_texts.append(c)
        corpus_meta.append((doc_id, d["name"], chunk_id))

# Compute embeddings
embs = []
for i in range(0, len(corpus_texts), EMBED_BATCH_SIZE):
    embs.extend(embedder.encode(corpus_texts[i:i+EMBED_BATCH_SIZE], normalize_embeddings=True, show_progress_bar=False))
embs = np.vstack(embs).astype('float32')

# FAISS index (cosine → use IndexFlatIP with normalized vectors)
index = faiss.IndexFlatIP(embs.shape[1])
index.add(embs)
print("Index built with", index.ntotal, "vectors.")

### 7.2 Summarization & generation pipelines

Load a summarization model (BART) and a general instruction-following
model (FLAN-T5) used for both summarization "reduce" steps and FAQ
question generation.


In [ ]:
# Summarization model

summarizer = pipeline(
    task="summarization",
    model=SUMM_MODEL,
    device=device_id
)


# QA / Generation model

generator = pipeline(
    task="text2text-generation",
    model=GEN_MODEL,
    device=device_id
)

### 7.3 Generation Quality Controls

 A set of defensive validation functions. These functions

*   detect prompt leakage
*   remove prompt echoes
* identify repetitive or corrupted text
* clean model responses

In [ ]:
# Detect prompt leakage
def is_prompt_leak(line: str) -> bool:
    return bool(PROMPT_LEAK_RE.search(line.strip()))


# Detect repetitive/degenerate generations
def has_repetition_loop(text: str, max_repeat: int = 3) -> bool:
    words = text.split()

    for i in range(len(words) - max_repeat):
        window = words[i:i + max_repeat]

        if len(set(w.lower() for w in window)) == 1:
            return True

    return False

# Remove prompt echoes from generated output
def strip_prompt_echo(raw_output: str, prompt: str) -> str:
    raw_output = raw_output.strip()

    if prompt and raw_output.startswith(prompt.strip()):
        raw_output = raw_output[len(prompt.strip()):].strip()

    return raw_output

## 8. Summarization

Summarize each document with a map-reduce approach:

1. **Map** — split the document into chunks and summarize each chunk with BART.
2. **Reduce** — join the partial summaries and ask FLAN-T5 to produce 7–10
   concise, fact-rich bullet points.

In [ ]:
def safe_truncate(txt, max_chars=2400):
    # keep a little room
    return txt if len(txt) <= max_chars else txt[:max_chars] + "…"

def summarize_long_text(text: str, chunk_chars: int = 1200, overlap_sentences: int = 1,
                         map_max_len=256, map_min_len=60) -> str:
    # MAP: summarize each chunk with BART
    parts = chunk_text(text, target_chars=chunk_chars, overlap_sent=overlap_sentences)
    partial_summaries = []
    for p in parts if parts else [text]:
        p_in = safe_truncate(p, 3500)
        result = summarizer(
              p_in,
              max_new_tokens=map_max_len,
              min_length=map_min_len,
              do_sample=False,
              repetition_penalty=1.3,
              no_repeat_ngram_size=3,
          )[0]

        out = result.get("generated_text", result.get("summary_text", "")).strip()
        partial_summaries.append(out.strip())
    # REDUCE: join partials and ask FLAN for crisp bullet points
    joined = safe_truncate(" ".join(partial_summaries), 4500)

    reduce_prompt = SUMMARIZATION_PROMPT.format(joined = joined)
    final = generator(reduce_prompt,
                      max_new_tokens=512,
                      temperature=0.2 ,
                      repetition_penalty=1.3,
                      no_repeat_ngram_size=3,)[0]["generated_text"]
    # Light cleanup: ensure bullets
    lines = [ln.strip() for ln in final.splitlines() if ln.strip()]
    bullets = []
    for ln in lines:
      if is_prompt_leak(ln):
        continue
      if has_repetition_loop(ln):
        continue
      if not ln.startswith("-"):
        ln = "- " + ln.lstrip("•*- ")
      bullets.append(ln)
    # Keep 7–10 bullets if possible
    if len(bullets) < 5:
        # fallback to original joined summary split
        bullets = ["- " + s for s in partial_summaries[:8]]
    return "\n".join(bullets[:10]).strip()

# Re-run to rebuild summaries with the new style
doc_summaries = {}
for d in docs_text:
    fname = d["name"]
    print(f"Summarizing: {name_to_title[fname]}")
    doc_summaries[fname] = summarize_long_text(d["text"])
print("Done.")

## 9. Retrieval (Semantic Search)

Given a free-text query, embed it with the same embedding model and search
the FAISS index for the most similar chunks, returning each hit's
similarity score, text, and source metadata.

In [ ]:

def search(query: str, top_k: int = 6) -> List[Tuple[float, str, Tuple[int,str,int]]]:
    # This converts the query text into an embedding vector (numbers representing meaning).
    qe = embedder.encode([query], normalize_embeddings=True)
    # This searches the FAISS index to find the closest vectors to the query vector.
          # D: distances/similarity scores for each match
          # I: indices (positions) of the matching vectors in the index
    D, I = index.search(qe.astype('float32'), top_k)
    hits = []
    for score, idx in zip(D[0], I[0]):
        hits.append((float(score), corpus_texts[idx], corpus_meta[idx]))  # (score, chunk_text, (doc_id, doc_name, chunk_id))
    return hits

## 10. FAQ Generation

Generate specific, grounded FAQ questions from each document's summary, then
filter out generic/low-value questions (e.g. *"What is the main point?"*)
using a simple heuristic.

### 10.1 Question-quality filter

In [ ]:
def looks_specific(q: str) -> bool:
    q = q.strip().rstrip("?")
    if len(q.split()) < 6:  # avoid very short
        return False
    if GENERIC_RE.search(q):
        return False
    # discourage ultra-vague starts
    if re.match(r"(?i)what is|what does|why is|who is$", q[:20]):
        return False
    return True

### 10.2 Question generation

`generate_varied_questions_from_summary` prompts FLAN-T5 (with sampling
enabled for variety) to produce a batch of candidate questions, deduplicates
them, filters out generic ones with `looks_specific`, and returns a shuffled
top-`k` selection.

In [ ]:
def dedup_preserve_order(items):
    seen = set(); out = []
    for x in items:
        key = re.sub(r"\s+", " ", x.strip().lower())
        if key in seen:
            continue
        seen.add(key); out.append(x.strip())
    return out

def generate_varied_questions_from_summary(summary_text: str, k: int = 12) -> list:
    base = safe_truncate(summary_text, 4500)
    prompt = FAQ_QS_PROMPT.format(doc=base)
    out = generator(
        prompt,
        max_new_tokens=256,
        temperature=0.9,       # sampling for variety
        top_p=0.9,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
    )[0]["generated_text"]
    qs = [q.strip().rstrip("?") + "?" for q in out.splitlines() if q.strip() and not is_prompt_leak(q) and not has_repetition_loop(q)]
    qs = dedup_preserve_order(qs)
    # filter generic
    qs = [q for q in qs if looks_specific(q)]
    # keep top ~10 and randomize a bit to avoid same 5 every run
    random.shuffle(qs)
    return qs[:max(k, 5)]

## 11. Single-Document Question Answering
Answer a question using only the chunks retrieved from one specific document. If retrieval comes back weak (too little context), fall back to the document's summary — and if the model still gives an empty or evasive answer, retry using the summary alone as context.

In [ ]:
def answer_within_doc_robust(question: str, doc_name: str, summary_text: str) -> str:
    # primary: restrict hits to this doc
    hits = search(question, top_k=12)
    hits = [h for h in hits if h[2][1] == doc_name]
    ctx_blocks, seen = [], set()
    for sc, txt, meta in sorted(hits, key=lambda x: -x[0]):
        if len(" ".join(ctx_blocks)) > 4200: break
        sig = hash(txt)
        if sig in seen:
            continue
        seen.add(sig); ctx_blocks.append(txt)

    # fallback: if retrieval is weak, include summary
    context = "\n\n".join(ctx_blocks)
    if len(context) < 400 and summary_text:
        # prepend summary to ensure we have something factual
        context = (summary_text.strip() + "\n\n" + context).strip()

    if not context:
        # absolute fallback to summary only (better than "Not specified")
        context = summary_text if summary_text else "No context."

    prompt = ANSWER_PROMPT_TMPL.format(context=context, question=question)
    ans = generator(
        prompt,
        max_new_tokens=220,
        temperature=0.2,   # deterministic-ish answers
        do_sample=False,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
    )[0]["generated_text"].strip()

    # last resort: if model still returns something empty or evasive, try a 2nd pass using only summary
    if (not ans) or len(ans.split()) < 3 or "I don't know" in ans:
        if summary_text:
            prompt2 = ANSWER_PROMPT_TMPL.format(context=summary_text, question=question)
            ans2 = generator(prompt2, max_new_tokens=200, temperature=0.2, do_sample=False , repetition_penalty=1.3 , no_repeat_ngram_size=3,)[0]["generated_text"].strip()
            if ans2 and "I don't know" not in ans2:
                ans = ans2

    # keep it tight
    return ans

## 12. FAQ Generation with Answers
Produce question–answer pairs for a single document: generate candidate questions from its summary, top up with synthesized questions from summary bullets if too few survive filtering, then answer each of the best 5 using answer_within_doc_robust.

In [ ]:
def generate_faqs_for_document(doc_name: str, title: str, text: str, summary: str) -> list:
    # ensure we have a seed summary; if not, synthesize a short one
    seed = summary if (summary and len(summary) > 60) else summarize_long_text(text)

    # 1) make 12 candidates from summary
    candidates = generate_varied_questions_from_summary(seed, k=12)

    # 2) if we somehow have <5, synthesize from top bullets
    if len(candidates) < 5:
        bullets = [ln[2:].strip() for ln in seed.splitlines() if ln.strip().startswith("- ")]
        for b in bullets:
            if len(candidates) >= 5: break
            frag = " ".join(b.split()[:12])
            candidates.append(f"What specific findings does the document report about {frag}?")
        candidates = dedup_preserve_order(candidates)

    # 3) pick 5 best-looking questions (stable order but de-generic)
    final_qs = candidates[:5]

    # 4) answer each question robustly
    faqs = []
    for q in final_qs:
        a = answer_within_doc_robust(q, doc_name, summary)
        faqs.append((q, a))
    return faqs

### 12.1 Run FAQ generation for every document

Apply `generate_faqs_for_document` to every uploaded document and store
the resulting question/answer pairs keyed by filename.

In [ ]:
doc_faqs = {}
for d in docs_text:
    fname = d["name"]
    print("Generating FAQs for", name_to_title[fname])
    doc_faqs[fname] = generate_faqs_for_document(fname, name_to_title[fname], d["text"], doc_summaries.get(fname))
print("Done (Improved FAQ Generation v2).")

## 13. Main Q&A Function
A general-purpose question-answering function that searches across all documents (not restricted to one, unlike answer_within_doc_robust), builds a deduplicated context from the top hits, generates an answer, and returns it alongside source pointers and retrieved snippets for transparency.

In [ ]:
def answer_question(question: str, top_k: int = 6, max_ctx_chars: int = 4200) -> Dict:
    hits = search(question, top_k=top_k)
    # merge contexts, sorted by score descending
    ctx_blocks = []
    seen = set()
    for sc, txt, meta in sorted(hits, key=lambda x: -x[0]):
        if len(" ".join(ctx_blocks)) > max_ctx_chars: break
        # avoid exact dupes
        sig = hash(txt)
        if sig in seen:
            continue
        seen.add(sig)
        ctx_blocks.append(txt)
    context = "\n\n".join(ctx_blocks)
    prompt = ANSWER_PROMPT_TMPL.format(context=context, question=question)
    out = generator(prompt, max_new_tokens=256, temperature=0.2,repetition_penalty=1.3,no_repeat_ngram_size=3,)[0]["generated_text"].strip()
    # include source pointers
    sources = [{"doc_name": m[1], "chunk_id": m[2], "score": float(s)} for (s,_,m) in hits[:min(top_k,5)]]
    return {"answer": out, "sources": sources, "retrieved_snippets": [t for _,t,_ in hits[:3]]}

## 14. Gradio Interface Helpers

Build a lookup from document title → filename and define helper functions used by the Gradio interface for question answering, summary retrieval, and FAQ access.

### 14.1 Question Answering Interface

Function used by the UI to answer user questions. It supports
*   document-specific retrieval
*   multi-document retrieval


In [ ]:
TITLE_TO_NAME = {name_to_title[d["name"]]: d["name"] for d in docs_text}

def qa_interface(question, title_scope):
    # map chosen title to original filename
    if title_scope != "All":  # checks if the user selected a specific document instead of searching all documents
        doc_scope = TITLE_TO_NAME[title_scope]
        # Restrict to that doc
        hits = search(question, top_k=12)
        hits = [h for h in hits if h[2][1] == doc_scope]
        ctx = []
        seen = set()
        for sc, txt, meta in sorted(hits, key=lambda x: -x[0]):
            if len(" ".join(ctx)) > 4200: break
            sig = hash(txt)
            if sig in seen: continue
            seen.add(sig); ctx.append(txt)
        context = "\n\n".join(ctx) if ctx else "No context available."
        prompt = ANSWER_PROMPT_TMPL.format(context=context, question=question)
        out = generator(prompt, max_new_tokens=256, temperature=0.2,repetition_penalty=1.3,no_repeat_ngram_size=3,)[0]["generated_text"].strip()
        sources = [{"title": name_to_title[m[1]], "chunk_id": m[2], "score": float(s)} for (s,_,m) in hits[:5]]
        return out, json.dumps(sources, indent=2)
    else:
        res = answer_question(question, top_k=8)
        # decorate sources with titles
        for s in res["sources"]:
            s["title"] = name_to_title.get(s["doc_name"], s["doc_name"])
            del s["doc_name"]
        return res["answer"], json.dumps(res["sources"], indent=2)





### 14.2 Content Retrieval Helpers

Define utility functions that retrieve summaries and FAQs precomputed document content for display in the interface.

In [ ]:
def get_summary(title_scope):
    if title_scope == "All":
        merged = []
        for d in docs_text:
            fname = d["name"]
            merged.append(f"### {name_to_title[fname]}\n{doc_summaries[fname]}")
        return "\n\n".join(merged)
    fname = TITLE_TO_NAME[title_scope]
    return doc_summaries.get(fname, "No summary available.")


def get_faqs(title_scope):
    if title_scope == "All":
        out = []
        for d in docs_text:
            fname = d["name"]
            friendly = name_to_title[fname]
            out.append("### " + friendly + "\n" + "\n".join([f"Q{i+1}: {q}\nA{i+1}: {a}" for i,(q,a) in enumerate(doc_faqs[fname])]))
        return "\n\n".join(out)
    fname = TITLE_TO_NAME[title_scope]
    faqs = doc_faqs.get(fname, [])
    return "\n".join([f"Q{i+1}: {q}\nA{i+1}: {a}" for i,(q,a) in enumerate(faqs)]) or "No FAQs."

## 15 Validation & Debugging

Optional checks to verify pipeline output without modifying any data.

### 15.1 Document Inspection

Review extracted text from loaded documents to verify successful parsing.


In [ ]:
for doc in docs_text:
    print("=" * 80)
    print(doc["name"])
    print(doc["text"][:1000])

### 15.2 Chunk Inspection

Display generated chunks and overlapping regions to verify chunking behavior.


In [ ]:
for doc in all_docs_chunks:
    print(f"\nFILE: {doc['name']}")
    print(f"Total Chunks: {len(doc['chunks'])}")

    for i, chunk in enumerate(doc["chunks"][:3]):
        print("\n" + "-" * 60)
        print(f"Chunk {i+1}")
        print(chunk[:500])


### 15.3 looks_specific() Smoke Tests

Basic tests to verify the behavior of the question-specificity filter.

In [ ]:
assert looks_specific("What is Artificial Intelligence (AI)?") is False, "Generic question incorrectly passed filter."
assert looks_specific("What specific outcomes did the 2021 pilot program report for District 4?") is True , \
    "Specific question incorrectly rejected by filter."

### 15.4 Retrieval Inspection

Run sample queries and inspect retrieved chunks with similarity scores.

In [ ]:
queries = [
    "What is artificial intelligence?",
    "Machine learning applications",
]

for q in queries:
    print(f"\nQUERY: {q}")

    results = search(q, top_k=3)

    for score, text, meta in results:
        print(f"\nScore: {score:.3f}")
        print(f"Source: {meta[1]}")
        print(text[:400])

### 15.5 Summary Inspection

Review generated document summaries.

In [ ]:
for fname, summary in doc_summaries.items():
    print("=" * 80)
    print(fname)
    print(summary)

### 15.6 FAQ Inspection

Review generated FAQs for each document.

In [ ]:
for fname, faqs in doc_faqs.items():
    print("=" * 80)
    print(fname)

    for i, (q, a) in enumerate(faqs, 1):
        print(f"\nQ{i}: {q}")
        print(f"A{i}: {a}")

## 16.Launch the Gradio App
### 16.1 Dropdown choices

Fix: the app below references DOC_TITLES for the scope dropdown, but it was never defined earlier in the notebook. Built here from TITLE_TO_NAME (Section 15), with "All" prepended.

In [ ]:
DOC_TITLES = ["All"] + list(TITLE_TO_NAME.keys())

## 16.2 Build the interface
Three tabs share one scope dropdown: Ask Questions (calls qa_interface), Summaries (calls get_summary), FAQs (calls get_faqs).

In [ ]:
with gr.Blocks(title="Mini NotebookLM (Local RAG)") as demo:
    gr.Markdown("# 📚 Mini NotebookLM (Local RAG) – Colab\nUpload ➜ Summaries & FAQs ➜ Ask Questions, grounded in your docs.")
    with gr.Row():
        title_scope = gr.Dropdown(DOC_TITLES, value="All", label="Document Scope (by Title)")
    with gr.Tab("Ask Questions"):
        question = gr.Textbox(label="Your question", placeholder="Ask something grounded in your uploaded docs…")
        btn = gr.Button("Answer")
        answer = gr.Textbox(label="Answer", lines=8)
        sources = gr.Textbox(label="Top Source Chunks (title, chunk_id, score)", lines=8)
        btn.click(qa_interface, inputs=[question, title_scope], outputs=[answer, sources])
    with gr.Tab("Summaries"):
        btn_sum = gr.Button("Show Summary")
        sum_box = gr.Textbox(label="Summary", lines=20)
        btn_sum.click(get_summary, inputs=[title_scope], outputs=[sum_box])
    with gr.Tab("FAQs"):
        btn_faq = gr.Button("Show FAQs (5)")
        faq_box = gr.Textbox(label="FAQs", lines=20)
        btn_faq.click(get_faqs, inputs=[title_scope], outputs=[faq_box])

### 16.3 Launch

In [ ]:
demo.launch(share=True , debug = True )